# Futu 原始逐分钟成交额：来源语义与Qlib时钟门

事前冻结原SOXL/SOXX `NONE` parquet、NAS snapshot/逐件SHA和Qlib全分钟provider。仅核原件 `turnover/volume` 是否为该bar交易价格范围内的报告成交额加权价，以及相对旧`close×volume`/`open×volume`代理是否有实质日内残差；不读取未来标签/账户结果。时间`t+1`送达仍是研究假定。

In [1]:
import os,json,hashlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
root=Path.cwd();card=root/'research/stage1_v5_raw_turnover_minute_source_20261030';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(root/plan['source_manifest'])==plan['source_manifest_sha256']
assert sha(root/plan['nas_binding'])==plan['nas_binding_sha256']
assert sha(root/plan['qlib_provider_uri']/'qlib_store_manifest.json')==plan['qlib_provider_manifest_sha256']
assert json.loads((card/'nas_original_readback.json').read_text())['status']=='passed'
manifest=json.loads((root/plan['source_manifest']).read_text());binding=json.loads((root/plan['nas_binding']).read_text())
assert binding['verified'] and binding['source_id']==plan['source_manifest_sha256']
for old in plan['old_routes']:assert (root/old).is_file()
qlib.init(provider_uri=str(root/plan['qlib_provider_uri']),region='us',kernels=1)
raw={}
for symbol,spec in plan['original_parquets'].items():
 path=root/spec['relative_path'];assert sha(path)==spec['sha256']
 x=pd.read_parquet(path,columns=['time_key','open','high','low','close','volume','turnover'])
 assert len(x)==spec['rows'];x['datetime']=pd.to_datetime(x.time_key)
 x=x.loc[x.datetime.between('2018-10-15 09:31:00','2022-12-22 16:00:00')].set_index('datetime').sort_index()
 assert x.index.is_unique
 raw[symbol]=x
print('verified source parquet rows',{k:len(v) for k,v in raw.items()})

[43278:MainThread](2026-10-30 16:11:08,357) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[43278:MainThread](2026-10-30 16:11:08,826) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[43278:MainThread](2026-10-30 16:11:08,827) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


verified source parquet rows {'SOXL': 410220, 'SOXX': 410220}


In [2]:
fields=['$raw_open','$raw_high','$raw_low','$raw_close','$raw_volume','$price_valid']
bars=D.features(['SOXL','SOXX'],fields,start_time='2018-10-15 09:31:00',end_time='2022-12-22 16:00:00',freq='1min')
assert len(bars)==820440
joined=[];max_volume_error=0.;max_price_error=0.
for symbol in ('SOXL','SOXX'):
 q=bars.xs(symbol,level='instrument').copy();source=raw[symbol]
 assert source.index.equals(q.index)
 x=source[['open','high','low','close','volume','turnover']].copy()
 for name in ('open','high','low','close'):
  max_price_error=max(max_price_error,float(np.nanmax(np.abs(x[name].to_numpy(float)-q['$raw_'+name].to_numpy(float)))))
 err=float(np.nanmax(np.abs(x.volume.to_numpy(float)-q['$raw_volume'].to_numpy(float))));max_volume_error=max(max_volume_error,err)
 x['price_valid']=q['$price_valid'].to_numpy(float);x['symbol']=symbol;x['trade_date']=x.index.normalize()
 joined.append(x)
source=pd.concat(joined).sort_index()
assert max_volume_error<=plan['gates']['maximum_raw_volume_qlib_absolute_error']
valid=source.price_valid.eq(1) & source.volume.gt(0) & source.turnover.gt(0)
v=source.loc[valid].copy();v['reported_vwap']=v.turnover/v.volume
v['ohlc4']=(v.open+v.high+v.low+v.close)/4
v['vwap_minus_close_bp']=10000*np.log(v.reported_vwap/v.close)
v['vwap_minus_ohlc4_bp']=10000*np.log(v.reported_vwap/v.ohlc4)
v['inside_ohlc_1bp']=(v.reported_vwap>=v.low*np.exp(-.0001))&(v.reported_vwap<=v.high*np.exp(.0001))
positive=(source.loc[source.price_valid.eq(1)&source.volume.gt(0),'turnover']>0).mean()
inside=float(v.inside_ohlc_1bp.mean())
perday=v.groupby(['symbol','trade_date']).agg(bars=('reported_vwap','size'),p90_abs_vwap_minus_ohlc4_bp=('vwap_minus_ohlc4_bp',lambda x:float(np.quantile(np.abs(x),.9))),median_abs_vwap_minus_close_bp=('vwap_minus_close_bp',lambda x:float(np.median(np.abs(x))))).reset_index();perday['year']=perday.trade_date.dt.year
annual=[]
for (symbol,year),g in perday.groupby(['symbol','year']):
 annual.append({'symbol':symbol,'year':int(year),'days':len(g),'bars':int(g.bars.sum()),'days_p90_abs_ohlc4_residual_gt_0_25bp':int(g.p90_abs_vwap_minus_ohlc4_bp.gt(.25).sum()),'median_day_p90_abs_ohlc4_residual_bp':float(g.p90_abs_vwap_minus_ohlc4_bp.median()),'median_day_median_abs_close_gap_bp':float(g.median_abs_vwap_minus_close_bp.median())})
annual=pd.DataFrame(annual).sort_values(['symbol','year'])
discovery=annual.loc[annual.year.isin(plan['discovery_years'])]
novel_days=int(discovery.days_p90_abs_ohlc4_residual_gt_0_25bp.sum()/2)
coverage=all(int(annual.loc[(annual.symbol==symbol)&(annual.year==2018),'days'].iloc[0])>=plan['gates']['minimum_2018_source_days'] and all(int(annual.loc[(annual.symbol==symbol)&(annual.year==y),'days'].iloc[0])>=plan['gates']['minimum_full_year_source_days'] for y in (2019,2020,2021,2022)) for symbol in ('SOXL','SOXX'))
passed=bool(positive>=plan['gates']['minimum_positive_turnover_fraction_when_volume_positive'] and inside>=plan['gates']['minimum_vwap_inside_raw_ohlc_fraction_with_1bp_tolerance'] and novel_days>=plan['gates']['minimum_discovery_days_with_daily_p90_abs_vwap_minus_ohlc4_gt_0_25bp'] and coverage)
old_hashes={old:sha(root/old) for old in plan['old_routes']}
audit={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'source_snapshot_id':binding['source_id'],'qlib_pair_rows':len(bars),'source_pair_rows':len(source),'max_raw_volume_qlib_error':max_volume_error,'max_raw_ohlc_qlib_error':max_price_error,'positive_reported_turnover_fraction_when_volume_positive':float(positive),'reported_vwap_inside_raw_ohlc_1bp_fraction':inside,'discovery_distinct_residual_days_pair_average':novel_days,'annual':annual.to_dict('records'),'old_route_conclusion_sha256':old_hashes,'source_gate_pass':passed,'first_historical_receipt_qualified':False,'new_model_fits':0,'new_candidate_accounts':0,'historical_research_passed':False,'decision':'reported_minute_turnover_source_gate_pass_needs_qlib_field_and_information' if passed else 'reported_minute_turnover_source_gate_failed'}
(card/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2)+'\n');annual.to_csv(card/'annual_source_summary.csv',index=False)
print(json.dumps({'decision':audit['decision'],'positive_turnover':positive,'inside_ohlc':inside,'novel_days':novel_days,'max_volume_error':max_volume_error,'annual':audit['annual']},ensure_ascii=False,indent=2))

{
  "decision": "reported_minute_turnover_source_gate_failed",
  "positive_turnover": 1.0,
  "inside_ohlc": 0.9077165427805977,
  "novel_days": 558,
  "max_volume_error": 0.0,
  "annual": [
    {
      "symbol": "SOXL",
      "year": 2018,
      "days": 53,
      "bars": 19913,
      "days_p90_abs_ohlc4_residual_gt_0_25bp": 53,
      "median_day_p90_abs_ohlc4_residual_bp": 10.32568403847606,
      "median_day_median_abs_close_gap_bp": 5.746593089646791
    },
    {
      "symbol": "SOXL",
      "year": 2019,
      "days": 252,
      "bars": 96012,
      "days_p90_abs_ohlc4_residual_gt_0_25bp": 252,
      "median_day_p90_abs_ohlc4_residual_bp": 6.5064635486691955,
      "median_day_median_abs_close_gap_bp": 3.127983678519729
    },
    {
      "symbol": "SOXL",
      "year": 2020,
      "days": 253,
      "bars": 98143,
      "days_p90_abs_ohlc4_residual_gt_0_25bp": 253,
      "median_day_p90_abs_ohlc4_residual_bp": 7.5934369777648785,
      "median_day_median_abs_close_gap_bp": 4.60988

## 中文结论

SOXX历史Goal未通过。原Futu SOXL/SOXX NONE分钟parquet逐件本地/NAS SHA一致，2018–2022共820440条与Qlib原价时钟一一匹配，原价成交量误差0，报告turnover在正量bar中正值率100%。然而以turnover/volume作为“同一分钟交易均价”仅90.77%落在放宽1bp的原价OHLC范围，低于冻结99.9%语义门；虽558个发现期日有区别于OHLC4的残差，也不能据此建新Qlib VWAP因子或拟合。官方Futu文档只称turnover为成交额，未证明与该OHLC同一成交过滤口径；真实首收未知。